In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
ART = ROOT / "artifacts"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from plotting import series_styles, new_figure

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

def es(v, d=4):
    """Numero con formato de Colombia: punto de miles y coma decimal."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}".replace(",", ".")
    s = f"{float(v):,.{d}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def tabla(df, d=4, enteros=()):
    out = df.copy()
    for c in out.columns:
        if c in enteros:
            out[c] = out[c].map(lambda v: es(int(v)) if pd.notna(v) else "")
        elif pd.api.types.is_float_dtype(out[c]):
            out[c] = out[c].map(lambda v: es(v, d))
        elif pd.api.types.is_integer_dtype(out[c]):
            out[c] = out[c].map(es)
    display(out.style.hide(axis="index"))

def figura(ruta, ancho=None):
    display(Image(filename=str(ruta), width=ancho))

# Proyecto Integrador 2.12: MLP para predecir clics en anuncios móviles

**Donny Torres** · Deep Learning · Maestría en Analítica de Datos, Universidad del Norte · noviembre de 2026

## Resumen

Este proyecto construye clasificadores MLP para predecir si un usuario hace clic en un anuncio móvil, compara su implementación en scikit-learn y en PySpark en modo local, e interpreta sus predicciones con LIME contrastado con SHAP, siguiendo el enunciado 2.12 del curso {cite}`enunciado212`. Se usan los 40.428.967 registros de Avazu, de 10 días de octubre de 2014, con una partición temporal fija: train días 21 a 28, validación día 29 y prueba día 30. Todos los modelos se entrenan con 5 semillas, con el ROC AUC como métrica primaria y un margen de equivalencia de $\delta = 0{,}01$ fijado antes de ver los resultados.

El mejor modelo es el MLP de scikit-learn, con una capa oculta de 50 neuronas y codificación combinada (hashing, frecuencia y target encoding fuera de pliegue), con un AUC de 0,7306 (IC 95 % de 0,7288 a 0,7325) y buena calibración. Su ventaja sobre las líneas base es pequeña: es equivalente a HistGB dentro de $\delta$ y supera a la regresión logística en 0,0041, una diferencia significativa pero menor que $\delta$. En Spark, el mejor modelo es la regresión logística entrenada con los 32,4 millones de filas de train (0,7157), y el MLP de Spark queda en 0,6607, 7,2 puntos por debajo del de scikit-learn, porque su implementación densifica la entrada y obligó a reducir la representación y la muestra.

En tiempo, scikit-learn fue más rápido en el flujo completo en todos los volúmenes en que pudo correr; en el ajuste del modelo, Spark lo alcanza entre unas 400 mil y 1 millón de filas, y desde 1,6 millones scikit-learn deja de caber en la memoria disponible. Spark local no aprovechó más de dos núcleos en este procesador, y la ley de Amdahl no describe su curva de escalamiento. En interpretabilidad, LIME fue estable pero solo moderadamente fiel ($R^2$ medio de 0,42), y su variable principal resultó un efecto de la discretización; SHAP y la importancia por permutación coinciden y señalan la tasa histórica de clic de la aplicación como la causa de los falsos positivos más confiados. Las ablaciones muestran que un hashing de 2^10 posiciones basta, que ni el cruce de variables ni una segunda capa aportan, y que la ponderación de clases no mejora el AUC y empeora la calibración.

## Cómo leer este libro

| Capítulo | Contenido |
|---|---|
| 1 | Ficha de datos, preguntas de investigación P1 a P4 e hipótesis H1 y H2 |
| 2 | Análisis exploratorio, con decisión, impacto y riesgo por hallazgo |
| 3 | Protocolo anti-fuga y preprocesamiento en ambos entornos |
| 4 y 5 | Modelado y ajuste de hiperparámetros en scikit-learn y en PySpark |
| 6 | Evaluación estadística: discriminación, calibración, umbral, lift, DeLong, McNemar y TOST |
| 7 | Benchmarking: tiempos por etapa, escalamiento, Amdahl, punto de cruce, Spark UI y costo |
| 8 | Interpretabilidad: LIME, fidelidad, estabilidad, sensibilidad, SHAP y permutación |
| 9 | Comparación de resultados y ablaciones |
| 10 | Reflexión crítica, respuestas a las preguntas, conclusiones y bibliografía |
| 11 | Anexo de reproducibilidad |

Todas las cifras del libro se leen de los archivos de `artifacts/`, que generan los scripts de `src/`; los cuadernos no vuelven a entrenar modelos.